In [10]:
from reachy_sdk import ReachySDK
from gtts import gTTS
import pygame, httpx
import socket
import io
import sounddevice as sd
import numpy as np
import scipy.io.wavfile as wav
import whisper
import numpy as np
from google import genai
from pydub import AudioSegment
reachy = ReachySDK(host='localhost')

In [2]:
#SPEAK function

def speak(text, lang='en', speed = 1.3): #speed = 1 is default
    tts = gTTS(text=text, lang=lang)
    buf = io.BytesIO()
    tts.write_to_fp(buf)
    buf.seek(0)
    
    pygame.mixer.init()
    pygame.mixer.music.load(buf, 'mp3')
    pygame.mixer.music.set_volume(3)  # 0.0 to 1.0
    pygame.mixer.music.play()
    while pygame.mixer.music.get_busy():
        pygame.time.Clock().tick(10)
        
        # tts = gTTS(
#     text="Hello, I am Reachy",
#     lang='en',        # language
#     tld='co.uk',      # accent! com - US English (default), co.uk - British, com.au- Australian,ca- Canadian,co.in- Indian
#     slow=False        # slow mode for clarity
# )

In [3]:
def get_respeaker_index():
    for i, dev in enumerate(sd.query_devices()):
        if 'ReSpeaker' in dev['name']:
            return i
    raise RuntimeError("ReSpeaker not found!")
DEVICE_INDEX = get_respeaker_index()  # dynamically finds ReSpeaker


print(f"DEVICE_INDEX: {DEVICE_INDEX}")
print(f"CHANNELS: {CHANNELS}")

DEVICE_INDEX: 6


NameError: name 'CHANNELS' is not defined

In [4]:
import sounddevice as sd

# def get_respeaker_index():
#     for i, dev in enumerate(sd.query_devices()):
#         if 'ReSpeaker' in dev['name']:
#             return i
#     raise RuntimeError("ReSpeaker not found!")
# DEVICE_INDEX = get_respeaker_index()  # dynamically finds ReSpeaker


# print(f"DEVICE_INDEX: {DEVICE_INDEX}")
# print(f"CHANNELS: {CHANNELS}")

SAMPLE_RATE = 16000
DEVICE_INDEX = 6
CHANNELS = 6
CHUNK_SIZE = 1024
SILENCE_THRESHOLD = 300    # adjust based on environment
SILENCE_DURATION = 1.5     # seconds of silence before stopping
MIN_SPEECH_DURATION = 0.5  # ignore very short sounds (coughs, clicks)

def record_until_silence():
    print("Listening...")
    chunks = []
    silent_chunks = 0
    speech_chunks = 0
    max_silent_chunks = int((SILENCE_DURATION * SAMPLE_RATE) / CHUNK_SIZE)
    min_speech_chunks = int((MIN_SPEECH_DURATION * SAMPLE_RATE) / CHUNK_SIZE)
    started_speaking = False

    with sd.InputStream(samplerate=SAMPLE_RATE, channels=CHANNELS,
                        dtype='int16', device=DEVICE_INDEX,
                        blocksize=CHUNK_SIZE) as stream:
        while True:
            chunk, _ = stream.read(CHUNK_SIZE)
            volume = np.abs(chunk[:, 0]).mean()  # use channel 0

            if volume > SILENCE_THRESHOLD:
                started_speaking = True
                silent_chunks = 0
                speech_chunks += 1
                chunks.append(chunk[:, 0])
            elif started_speaking:
                chunks.append(chunk[:, 0])
                silent_chunks += 1
                if silent_chunks >= max_silent_chunks:
                    # only accept if speech was long enough
                    if speech_chunks >= min_speech_chunks:
                        break
                    else:
                        # too short, reset and keep listening
                        chunks = []
                        silent_chunks = 0
                        speech_chunks = 0
                        started_speaking = False

    print("Done.")
    return np.concatenate(chunks, axis=0)

In [5]:
#Setup Whisper STT/ transcribe method
# import whisper
# import numpy as np

model = whisper.load_model("base")  # loads once

def transcribe(audio):
    # convert from int16 to float32 that whisper expects
    audio_float = audio.astype(np.float32) / 32768.0
    audio_flat = audio_float.flatten()  # whisper expects 1D array
    result = model.transcribe(audio_flat, fp16=False)
    return result["text"]

In [24]:
# #gestures
# make_movement:
from reachy_sdk.trajectory import goto
from reachy_sdk.trajectory.interpolation import InterpolationMode
reachy.turn_on('l_arm')
reachy.turn_on('r_arm')
import reachy_sdk.trajectory

right_hand_hello = {
    reachy.r_arm.r_shoulder_pitch: -8,
    reachy.r_arm.r_shoulder_roll: -12,
    reachy.r_arm.r_arm_yaw: -43,
    reachy.r_arm.r_elbow_pitch: -129,
    reachy.r_arm.r_forearm_yaw: -62,
    reachy.r_arm.r_wrist_pitch: -6,
    reachy.r_arm.r_wrist_roll: 42,
    reachy.r_arm.r_gripper: 12,
}

l_arm_relax = {
    reachy.l_arm.l_shoulder_pitch: -2,
    reachy.l_arm.l_shoulder_roll: 0,
    reachy.l_arm.l_arm_yaw: -41,
    reachy.l_arm.l_elbow_pitch: -3,
    reachy.l_arm.l_forearm_yaw: -57,
    reachy.l_arm.l_wrist_pitch: 5,
    reachy.l_arm.l_wrist_roll: 16,
    reachy.l_arm.l_gripper: -14,
}
movements = [right_hand_hello, r_arm_relax]
for moves in movements:
    goto(
        goal_positions = moves,
        duration = 6.0,
        interpolation_mode=InterpolationMode.MINIMUM_JERK,
        )


# goto(
#     goal_positions = right_hand_hello,
#     duration = 6.0,
#     interpolation_mode=InterpolationMode.MINIMUM_JERK,
# )

reachy.turn_off_smoothly('l_arm')
reachy.turn_off_smoothly('r_arm')

In [6]:

#Setting up Gemini LLM
with open('Gemini_key.env', 'r') as f:
    env_content = f.read()
    Gemini_key = env_content.split("'")[1]
instruction = f"""You are Reachy, a friendly humanoid robot being used for medical triage research in the IRON lab at the
 University of North Carolina at Chapel Hill. Keep responses short and conversational.Do not use emojis or special characters
in your responses."""

from google.genai import types
client = genai.Client(api_key=Gemini_key)

chat_session = client.chats.create(
    model="gemini-3.1-flash-lite-preview",
    config=types.GenerateContentConfig(
        system_instruction= instruction
    )
)

def chat(user_input):
    response = chat_session.send_message(user_input)
#     if "hello" in response.lower():
#         make_movement()
    return response.text


# with open('ChatGPT.env', 'r') as f:
#     env_content = f.read()
#     GPT_key = env_content.split("'")[1]

# from openai import OpenAI
# client = OpenAI(api_key=GPT_key)

# instruction = """You are Reachy, a friendly humanoid robot being used for medical triage research in the IRON lab at the
# University of North Carolina at Chapel Hill. Keep responses short and conversational. Do not use emojis or special characters
# in your responses."""

# def chat(user_input):
#     response = client.chat.completions.create(
#         model="gpt-4o-mini",
#         messages=[
#             {"role": "system", "content": instruction},
#             {"role": "user", "content": user_input}
#         ]
#     )
#     return response.choices[0].message.content

In [12]:
# Full loop for talking back and forth
while True:
    audio = record_until_silence()
    text = transcribe(audio)
    print(f"You: {text}")

    reply = chat(text)
    print(f"Reachy: {reply}")

    speak(reply)


Listening...


KeyboardInterrupt: 

In [ ]:
#YOLO
from ultralytics import YOLO
model = YOLO("yolov8n.pt")   # downloads ~6MB on first run automatically

def detect_objects(frame):
    results = model(frame, verbose=False)[0]
    return results.plot()    # returns frame with boxes+labels drawn

from reachy_sdk import ReachySDK
from http.server import BaseHTTPRequestHandler, HTTPServer
import cv2
import numpy as np
import threading
import urllib.request
import os

# ── Download model weights if not present ──────────────────────────────────────
MODEL_DIR = "model"
os.makedirs(MODEL_DIR, exist_ok=True)

PROTO   = f"{MODEL_DIR}/MobileNetSSD_deploy.prototxt"
WEIGHTS = f"{MODEL_DIR}/MobileNetSSD_deploy.caffemodel"

if not os.path.exists(PROTO):
    print("Downloading prototxt...")
    urllib.request.urlretrieve(
        "https://raw.githubusercontent.com/chuanqi305/MobileNet-SSD/master/MobileNetSSD_deploy.prototxt",
        PROTO
    )
if not os.path.exists(WEIGHTS):
    print("Downloading weights (~23MB)...")
    urllib.request.urlretrieve(
        "https://drive.google.com/uc?export=download&id=0B3gersZ2cHIxRm5PMWRoTkdHdHc",
        WEIGHTS
    )

CLASSES = [
    "background", "aeroplane", "bicycle", "bird", "boat", "bottle",
    "bus", "car", "cat", "chair", "cow", "diningtable", "dog",
    "horse", "motorbike", "person", "pottedplant", "sheep",
    "sofa", "train", "tvmonitor"
]

# Distinct color per class
COLORS = np.random.default_rng(42).integers(100, 255, size=(len(CLASSES), 3)).tolist()

net = cv2.dnn.readNetFromCaffe(PROTO, WEIGHTS)
print("Model loaded.")

# ── Reachy init ────────────────────────────────────────────────────────────────
reachy = ReachySDK(host='localhost')
reachy.left_camera.wait_for_new_frame()
reachy.right_camera.wait_for_new_frame()
print("Cameras ready, starting stream...")


# ── Frame builders ─────────────────────────────────────────────────────────────
def get_detection_frame():
    reachy.left_camera.wait_for_new_frame()
    frame = reachy.left_camera.last_frame.copy()
    frame = detect_objects(frame)
    cv2.putText(frame, "OBJECT DETECTION",
                (10, frame.shape[0] - 10),
                cv2.FONT_HERSHEY_SIMPLEX, 0.5, (0, 255, 136), 1)
    return frame

def get_stereo_detection_frame():
    """Side-by-side with detection on both cameras."""
    reachy.left_camera.wait_for_new_frame()
    reachy.right_camera.wait_for_new_frame()
    left  = detect_objects(reachy.left_camera.last_frame.copy())
    right = detect_objects(reachy.right_camera.last_frame.copy())
    cv2.putText(left,  'LEFT',  (10, 20), cv2.FONT_HERSHEY_SIMPLEX, 0.6, (0, 255, 100), 2)
    cv2.putText(right, 'RIGHT', (10, 20), cv2.FONT_HERSHEY_SIMPLEX, 0.6, (0, 180, 255), 2)
    return np.hstack([left, right])

# ── HTTP server ────────────────────────────────────────────────────────────────
class MJPEGHandler(BaseHTTPRequestHandler):
    def do_GET(self):
        if self.path in ('/stream', '/stream/stereo'):
            self.send_response(200)
            self.send_header('Content-type', 'multipart/x-mixed-replace; boundary=frame')
            self.end_headers()
            grab = get_stereo_detection_frame 
#             if 'stereo' in self.path else get_detection_frame
            try:
                while True:
                    frame = grab()
                    _, jpg = cv2.imencode('.jpg', frame, [cv2.IMWRITE_JPEG_QUALITY, 80])
                    self.wfile.write(b'--frame\r\n')
                    self.wfile.write(b'Content-Type: image/jpeg\r\n\r\n')
                    self.wfile.write(jpg.tobytes())
                    self.wfile.write(b'\r\n')
            except (BrokenPipeError, ConnectionResetError):
                pass

        elif self.path == '/':
            self.send_response(200)
            self.send_header('Content-type', 'text/html')
            self.end_headers()
            self.wfile.write(b"""
            <html><head><title>Reachy Detection</title>
            <style>
                body  { background:#111; color:#eee; font-family:sans-serif;
                        display:flex; flex-direction:column;
                        align-items:center; padding:20px; margin:0; }
                h1    { color:#00ff88; margin-bottom:12px; }
                nav   { margin-bottom:16px; }
                nav a { color:#00ccff; margin:0 12px; text-decoration:none;
                        font-size:1.1em; }
                img   { border:2px solid #00ff88; max-width:100%; }
            </style></head>
            <body>
                <h1>Reachy's Object Detection</h1>
                
                <img src="/stream" id="feed"
                     onerror="setTimeout(()=>location.reload(),1000)">
                <script>
                  const p = new URLSearchParams(location.search);
                  if (p.get('view') === 'stereo')
                      document.getElementById('feed').src = '/stream/stereo';
                </script>
            </body></html>""")
        else:
            self.send_response(404)
            self.end_headers()

    def log_message(self, *args):
        pass

server = HTTPServer(('0.0.0.0', 8080), MJPEGHandler)
print("Stream live at http://localhost:8080")
threading.Thread(target=server.serve_forever, daemon=True).start()
input("Press Enter to stop...\n")

In [ ]:
#MIC TEST
import sounddevice as sd
# import numpy as np
# import scipy.io.wavfile as wav

# ReSpeaker device is No. 6. Use this to double check
print(sd.query_devices())

   0 ReSpeaker 4 Mic Array (UAC1.0): USB Audio (hw:0,0), ALSA (6 in, 2 out)
   1 C2: USB Audio (hw:1,0), ALSA (2 in, 0 out)
   2 HDA Intel PCH: HDMI 0 (hw:2,3), ALSA (0 in, 8 out)
   3 HDA Intel PCH: HDMI 1 (hw:2,7), ALSA (0 in, 2 out)
   4 HDA Intel PCH: HDMI 2 (hw:2,8), ALSA (0 in, 8 out)
   5 HDA Intel PCH: HDMI 3 (hw:2,9), ALSA (0 in, 8 out)
   6 HDA Intel PCH: HDMI 4 (hw:2,10), ALSA (0 in, 8 out)
   7 C2: USB Audio (hw:3,0), ALSA (2 in, 0 out)
   8 sysdefault, ALSA (128 in, 128 out)
   9 front, ALSA (0 in, 2 out)
  10 surround40, ALSA (0 in, 2 out)
  11 iec958, ALSA (0 in, 2 out)
  12 spdif, ALSA (6 in, 2 out)
  13 pulse, ALSA (32 in, 32 out)
  14 upmix, ALSA (8 in, 8 out)
  15 vdownmix, ALSA (6 in, 6 out)
  16 dmix, ALSA (0 in, 2 out)
* 17 default, ALSA (32 in, 32 out)


In [ ]:
#yell out its IP
#import socket

def get_ip():
    s = socket.socket(socket.AF_INET, socket.SOCK_DGRAM)
    try:
        s.connect(("8.8.8.8", 80))  # doesn't actually send data
        return s.getsockname()[0]
    finally:
        s.close()

print(get_ip())

152.23.165.203


In [ ]:
# Speaking TEST
speak("Hello, how are you today")

In [ ]:

sd.play(audio, SAMPLE_RATE)
sd.wait()

In [ ]:
DEVICE_INDEX = 0   # ReSpeaker's index from list of devices
SAMPLE_RATE = 16000
DURATION =  5      # seconds
CHANNELS = 6  # ReSpeaker has 4 mics

def record(duration=DURATION):
    print("Recording...")
    audio = sd.rec(
        int(duration * SAMPLE_RATE),
        samplerate=SAMPLE_RATE,
        channels=CHANNELS,
        dtype='int16',
        device=DEVICE_INDEX
    )
    sd.wait()
    print("Done.")
    return audio[:, 0]  # take only channel 0 (already beamformed by DSP)

#audio = record()


In [ ]:
#STT
#audio = record()
#text = transcribe(audio)
#print(text)
#speak(text)